In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as opt
from torch.utils.data import DataLoader,TensorDataset
from torch.nn.utils.rnn import pad_packed_sequence,pack_padded_sequence
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from sklearn import metrics
from sklearn.model_selection import train_test_split
import scipy.io as sio
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# =========================
# Full Reproducible Pipeline
# FASTA (pos_/neg_) -> BLOSUM62(1002x20) + FEGS(578) -> DVIB Train/Test
# =========================

import os
import math
import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as opt

from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn import metrics

# -------------------------
# 0) Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


# -------------------------
# 1) FASTA reader (pos_/neg_ labels)
# -------------------------
def read_fasta_with_labels(fasta_path: str):
    """
    Expected headers like:
        >pos_0
        SEQUENCE...
        >neg_1
        SEQUENCE...
    Label rule:
        header startswith 'pos' -> 1
        header startswith 'neg' -> 0
    """
    names, seqs, labels = [], [], []
    name = None
    seq_chunks = []

    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                # flush previous
                if name is not None:
                    seq = "".join(seq_chunks).upper()
                    if len(seq) == 0:
                        raise ValueError(f"Empty sequence for {name} in {fasta_path}")
                    names.append(name)
                    seqs.append(seq)
                    labels.append(infer_label_from_header(name))
                # start new
                name = line[1:].strip()
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())

    # flush last
    if name is not None:
        seq = "".join(seq_chunks).upper()
        if len(seq) == 0:
            raise ValueError(f"Empty sequence for {name} in {fasta_path}")
        names.append(name)
        seqs.append(seq)
        labels.append(infer_label_from_header(name))

    return names, seqs, np.array(labels, dtype=np.int64)


def infer_label_from_header(header: str) -> int:
    h = header.lower()
    if h.startswith("pos"):
        return 1
    if h.startswith("neg"):
        return 0
    # fallback: sometimes header ends with |0 or |1
    if "|" in h:
        tail = h.split("|")[-1].strip()
        if tail.isdigit():
            v = int(tail)
            if v in (0, 1):
                return v
    raise ValueError(
        f"Cannot infer label from header: '{header}'. "
        "Expected pos_/neg_ or something like ...|0 / ...|1"
    )


# -------------------------
# 2) BLOSUM62 encoding (like your get_blosum62 + padding to 1002)
# -------------------------
AA20 = "ACDEFGHIKLMNPQRSTVWY"
AA_INDEX = {a: i for i, a in enumerate(AA20)}

# Standard BLOSUM62 matrix for the 20 AAs in AA20 order above.
# Source values are the canonical BLOSUM62 scores.
BLOSUM62 = np.array([
#  A  C  D  E  F  G  H  I  K  L  M  N  P  Q  R  S  T  V  W  Y
  [ 4, 0,-2,-1,-2, 0,-2,-1,-1,-1,-1,-2,-1,-1,-1, 1, 0, 0,-3,-2], # A
  [ 0, 9,-3,-4,-2,-3,-3,-1,-3,-1,-1,-3,-3,-3,-3,-1,-1,-1,-2,-2], # C
  [-2,-3, 6, 2,-3,-1,-1,-3,-1,-4,-3, 1,-1, 0,-2, 0,-1,-3,-4,-3], # D
  [-1,-4, 2, 5,-3,-2, 0,-3, 1,-3,-2, 0,-1, 2, 0, 0,-1,-2,-3,-2], # E
  [-2,-2,-3,-3, 6,-3,-1, 0,-3, 0, 0,-3,-4,-3,-3,-2,-2,-1, 1, 3], # F
  [ 0,-3,-1,-2,-3, 6,-2,-4,-2,-4,-3, 0,-2,-2,-2, 0,-2,-3,-2,-3], # G
  [-2,-3,-1, 0,-1,-2, 8,-3,-1,-3,-2, 1,-2, 0, 0,-1,-2,-3,-2, 2], # H
  [-1,-1,-3,-3, 0,-4,-3, 4,-3, 2, 1,-3,-3,-3,-3,-2,-1, 3,-3,-1], # I
  [-1,-3,-1, 1,-3,-2,-1,-3, 5,-2,-1, 0,-1, 1, 2, 0,-1,-2,-3,-2], # K
  [-1,-1,-4,-3, 0,-4,-3, 2,-2, 4, 2,-3,-3,-2,-2,-2,-1, 1,-2,-1], # L
  [-1,-1,-3,-2, 0,-3,-2, 1,-1, 2, 5,-2,-2, 0,-1,-1,-1, 1,-1,-1], # M
  [-2,-3, 1, 0,-3, 0, 1,-3, 0,-3,-2, 6,-2, 0, 0, 1, 0,-3,-4,-2], # N
  [-1,-3,-1,-1,-4,-2,-2,-3,-1,-3,-2,-2, 7,-1,-2,-1,-1,-2,-4,-3], # P
  [-1,-3, 0, 2,-3,-2, 0,-3, 1,-2, 0, 0,-1, 5, 1, 0,-1,-2,-2,-1], # Q
  [-1,-3,-2, 0,-3,-2, 0,-3, 2,-2,-1, 0,-2, 1, 5,-1,-1,-3,-3,-2], # R
  [ 1,-1, 0, 0,-2, 0,-1,-2, 0,-2,-1, 1,-1, 0,-1, 4, 1,-2,-3,-2], # S
  [ 0,-1,-1,-1,-2,-2,-2,-1,-1,-1,-1, 0,-1,-1,-1, 1, 5, 0,-2,-2], # T
  [ 0,-1,-3,-2,-1,-3,-3, 3,-2, 1, 1,-3,-2,-2,-3,-2, 0, 4,-3,-1], # V
  [-3,-2,-4,-3, 1,-2,-2,-3,-3,-2,-1,-4,-4,-2,-3,-3,-2,-3,11, 2], # W
  [-2,-2,-3,-2, 3,-3, 2,-1,-2,-1,-1,-2,-3,-1,-2,-2,-2,-1, 2, 7], # Y
], dtype=np.float32)

def get_blosum62(seq: str, max_len: int = 1002) -> np.ndarray:
    """
    Returns array shape [max_len, 20]
    Each position gets the BLOSUM62 row vector for that AA.
    Pads with zeros, truncates to max_len.
    Unknown AAs -> zeros.
    """
    seq = seq.upper()
    out = np.zeros((max_len, 20), dtype=np.float32)
    L = min(len(seq), max_len)
    for i in range(L):
        aa = seq[i]
        if aa in AA_INDEX:
            out[i, :] = BLOSUM62[AA_INDEX[aa]]
    return out


# -------------------------
# 3) FEGS 578 = Vg158 + AAC20 + DPC400
# -------------------------

def aac_20(seq: str) -> np.ndarray:
    seq = seq.upper()
    L = len(seq)
    if L == 0:
        return np.zeros(20, dtype=np.float32)
    return np.array([seq.count(a)/L for a in AA20], dtype=np.float32)

def dpc_400(seq: str) -> np.ndarray:
    seq = seq.upper()
    total = len(seq) - 1
    if total <= 0:
        return np.zeros(400, dtype=np.float32)
    dipeptides = [a+b for a in AA20 for b in AA20]
    # simple count-based DPC
    return np.array([seq.count(dp)/total for dp in dipeptides], dtype=np.float32)


# A set of amino-acid properties (20-length vectors in AA20 order).
# These are standard, stable biochemical descriptors.
PROPS = {
    # Kyte-Doolittle hydrophobicity
    "hydro_kd": np.array([1.8,2.5,-3.5,-3.5,2.8,-0.4,-3.2,4.5,-3.9,3.8,1.9,-3.5,-1.6,-3.5,-4.5,-0.8,-0.7,4.2,-0.9,-1.3], dtype=np.float32),
    # Hopp-Woods hydrophilicity
    "hydrophil_hw": np.array([-0.5,-1.0,3.0,3.0,-2.5,0.0,-0.5,-1.8,3.0,-1.8,-1.3,0.2,0.0,0.2,3.0,0.3,-0.4,-1.5,-3.4,-2.3], dtype=np.float32),
    # Molecular weight (approx)
    "mw": np.array([89.09,121.16,133.10,147.13,165.19,75.07,155.16,131.18,146.19,131.18,149.21,132.12,115.13,146.15,174.20,105.09,119.12,117.15,204.23,181.19], dtype=np.float32),
    # Volume (approx, A^3)
    "vol": np.array([88.6,108.5,111.1,138.4,189.9,60.1,153.2,166.7,168.6,166.7,162.9,114.1,112.7,143.8,173.4,89.0,116.1,140.0,227.8,193.6], dtype=np.float32),
    # Polarity (Grantham)
    "polarity": np.array([8.1,5.5,13.0,12.3,5.2,9.0,10.4,5.2,11.3,4.9,5.7,11.6,8.0,10.5,10.5,9.2,8.6,5.9,5.4,6.2], dtype=np.float32),
    # Flexibility (approx)
    "flex": np.array([0.357,0.346,0.511,0.497,0.314,0.544,0.323,0.462,0.466,0.365,0.295,0.463,0.509,0.493,0.529,0.507,0.444,0.386,0.305,0.420], dtype=np.float32),
    # Isoelectric point pI (approx)
    "pI": np.array([6.01,5.07,2.77,3.22,5.48,5.97,7.59,6.02,9.74,5.98,5.74,5.41,6.30,5.65,10.76,5.68,5.60,5.96,5.89,5.66], dtype=np.float32),
    # Net charge at neutral pH (rough)
    "charge": np.array([0,0,-1,-1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0], dtype=np.float32),
}

def _zscore(v: np.ndarray) -> np.ndarray:
    v = v.astype(np.float32)
    mu = float(v.mean())
    sd = float(v.std() + 1e-8)
    return (v - mu) / sd

# normalize properties so scales are compatible
PROPS_Z = {k: _zscore(v) for k, v in PROPS.items()}

def _moments(x: np.ndarray) -> np.ndarray:
    """Return mean, var, skew, kurt (4,) with numeric stability."""
    x = x.astype(np.float32)
    if x.size == 0:
        return np.zeros(4, dtype=np.float32)
    mu = x.mean()
    var = ((x - mu) ** 2).mean()
    sd = math.sqrt(float(var) + 1e-8)
    skew = (((x - mu) / (sd + 1e-8)) ** 3).mean()
    kurt = (((x - mu) / (sd + 1e-8)) ** 4).mean()
    return np.array([mu, var, skew, kurt], dtype=np.float32)

def vg_158(seq: str) -> np.ndarray:
    """
    Build 158-D graphical descriptor:
    - Use 19 property pairs -> each yields 8 moments (x:4, y:4) => 152
    - Add 6 global radial/path stats => 6
    Total 158
    """
    seq = seq.upper()
    # map AA -> index; unknown -> skip as 0 contribution
    idxs = [AA_INDEX.get(a, None) for a in seq]

    # choose 19 pairs (can repeat props; objective is fixed-dim reproducible feature)
    pairs = [
        ("hydro_kd", "polarity"),
        ("mw", "hydro_kd"),
        ("vol", "hydro_kd"),
        ("pI", "charge"),
        ("flex", "hydro_kd"),
        ("hydrophil_hw", "hydro_kd"),
        ("mw", "polarity"),
        ("vol", "polarity"),
        ("pI", "hydro_kd"),
        ("pI", "polarity"),
        ("charge", "hydro_kd"),
        ("charge", "polarity"),
        ("flex", "polarity"),
        ("flex", "mw"),
        ("flex", "vol"),
        ("hydrophil_hw", "polarity"),
        ("hydrophil_hw", "mw"),
        ("hydrophil_hw", "vol"),
        ("mw", "pI"),
    ]  # 19 pairs

    feats = []

    # compute cumulative walk points for each pair
    for px, py in pairs:
        vx = PROPS_Z[px]
        vy = PROPS_Z[py]

        steps_x = []
        steps_y = []
        for ii in idxs:
            if ii is None:
                steps_x.append(0.0)
                steps_y.append(0.0)
            else:
                steps_x.append(float(vx[ii]))
                steps_y.append(float(vy[ii]))

        steps_x = np.array(steps_x, dtype=np.float32)
        steps_y = np.array(steps_y, dtype=np.float32)

        cx = np.cumsum(steps_x)
        cy = np.cumsum(steps_y)

        feats.append(_moments(cx))  # 4
        feats.append(_moments(cy))  # 4

    vg152 = np.concatenate(feats, axis=0)  # 19 * 8 = 152

    # 6 global stats from radius/path
    if len(idxs) == 0:
        extra6 = np.zeros(6, dtype=np.float32)
    else:
        # reuse a canonical walk (hydro_kd, polarity)
        vx = PROPS_Z["hydro_kd"]
        vy = PROPS_Z["polarity"]
        sx, sy = [], []
        for ii in idxs:
            if ii is None:
                sx.append(0.0); sy.append(0.0)
            else:
                sx.append(float(vx[ii])); sy.append(float(vy[ii]))
        cx = np.cumsum(np.array(sx, dtype=np.float32))
        cy = np.cumsum(np.array(sy, dtype=np.float32))
        r = np.sqrt(cx**2 + cy**2)

        # path length
        if len(cx) >= 2:
            dx = np.diff(cx); dy = np.diff(cy)
            path_len = float(np.sum(np.sqrt(dx*dx + dy*dy)))
        else:
            path_len = 0.0
        end_to_end = float(r[-1]) if r.size > 0 else 0.0

        rm = _moments(r)  # 4
        extra6 = np.array([path_len, end_to_end, rm[0], rm[1], rm[2], rm[3]], dtype=np.float32)

    out = np.concatenate([vg152, extra6], axis=0)
    assert out.shape[0] == 158, out.shape
    return out.astype(np.float32)

def fegs_578(seq: str) -> np.ndarray:
    vg = vg_158(seq)          # 158
    a = aac_20(seq)           # 20
    d = dpc_400(seq)          # 400
    fv = np.concatenate([vg, a, d], axis=0)  # 578
    return fv.astype(np.float32)


# -------------------------
# 4) Build tensors from FASTA
# -------------------------
def make_tensors_from_fasta(fasta_path: str, max_len: int = 1002, save_mat_path: str = None):
    names, seqs, labels = read_fasta_with_labels(fasta_path)

    N = len(seqs)
    evolution = torch.zeros(N, max_len, 20, dtype=torch.float32)
    fegs = torch.zeros(N, 578, dtype=torch.float32)
    lengths = torch.zeros(N, dtype=torch.long)

    for i, s in enumerate(seqs):
        lengths[i] = min(len(s), max_len)  # important for pack
        evolution[i, :, :] = torch.from_numpy(get_blosum62(s, max_len=max_len))
        fegs[i, :] = torch.from_numpy(fegs_578(s))

    labels_t = torch.tensor(labels, dtype=torch.long)

    # Optional: save MAT like the original benchmark files
    if save_mat_path is not None:
        os.makedirs(os.path.dirname(save_mat_path), exist_ok=True)
        sio.savemat(save_mat_path, {"FV": fegs.numpy()})

    return evolution, lengths, fegs, labels_t


# -------------------------
# 5) Model (DVIB) - keep paper architecture, but fix logits output
# -------------------------
class dvib(nn.Module):
    def __init__(self, k, out_channels, hidden_size):
        super(dvib, self).__init__()

        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=out_channels,
            kernel_size=(1, 20),
            stride=(1, 1),
            padding=(0, 0),
        )

        self.rnn = nn.GRU(
            input_size=out_channels,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )

        self.fc1 = nn.Linear(hidden_size * 4, hidden_size * 4)

        # paper concat: [cnn_vectors (hidden*4), FEGS (578)]
        self.enc_mean = nn.Linear(hidden_size * 4 + 578, k)
        self.enc_std  = nn.Linear(hidden_size * 4 + 578, k)

        self.dec = nn.Linear(k, 2)

        self.drop_layer = nn.Dropout(0.5)

        nn.init.xavier_uniform_(self.fc1.weight); nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_mean.weight); nn.init.constant_(self.enc_mean.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_std.weight);  nn.init.constant_(self.enc_std.bias, 0.0)
        nn.init.xavier_uniform_(self.dec.weight);      nn.init.constant_(self.dec.bias, 0.0)

    def cnn_gru(self, x, lens):
        # x: [B, 1002, 20]
        x = x.unsqueeze(1)          # [B,1,1002,20]
        x = self.conv(x)            # [B,C,1002,1]
        x = torch.relu(x)
        x = x.squeeze(3)            # [B,C,1002]
        x = x.permute(0, 2, 1)      # [B,1002,C]

        # pack_padded_sequence needs CPU lengths and sorted if enforce_sorted=True
        lens_cpu = lens.detach().cpu()
        gru_input = pack_padded_sequence(x, lens_cpu, batch_first=True, enforce_sorted=True)

        output, hidden = self.rnn(gru_input)

        # hidden: [num_layers*2, B, hidden_size] -> here 4,B,H
        output_all = torch.cat([hidden[-1], hidden[-2], hidden[-3], hidden[-4]], dim=1)  # [B, 4H]
        return output_all

    def forward(self, pssm, lengths, FEGS):
        cnn_vectors = self.cnn_gru(pssm, lengths)                  # [B, 4H]
        feature_vec = torch.cat([cnn_vectors, FEGS], dim=1)        # [B, 4H+578]

        enc_mean = self.enc_mean(feature_vec)
        enc_std  = F.softplus(self.enc_std(feature_vec) - 5)

        eps = torch.randn_like(enc_std)
        latent = enc_mean + enc_std * eps

        logits = self.dec(latent)   # IMPORTANT: logits for CrossEntropyLoss
        return logits, enc_mean, enc_std, latent


# -------------------------
# 6) Loss
# -------------------------
CE = nn.CrossEntropyLoss(reduction="sum")
betas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]

def calc_loss(logits, labels, enc_mean, enc_std, beta=1e-3):
    """
    logits: [B,2]
    labels: [B]
    """
    ce = CE(logits, labels)
    KL = 0.5 * torch.sum(enc_mean.pow(2) + enc_std.pow(2) - 2 * enc_std.log() - 1)
    return (ce + beta * KL) / logits.shape[0]


# -------------------------
# 7) Train / Test
# -------------------------
def run_train_test(
    train_fasta: str,
    test_fasta: str,
    batch_size: int = 100,
    out_channels: int = 128,
    hidden_size: int = 512,
    k: int = 1024,
    beta_idx: int = 2,
    lr: float = 1e-4,
    num_epochs: int = 50,
    save_mat_dir: str = None
):
    # Build tensors
    train_mat = os.path.join(save_mat_dir, "train_FEGS.mat") if save_mat_dir else None
    test_mat  = os.path.join(save_mat_dir, "test_FEGS.mat")  if save_mat_dir else None

    train_pssm, train_len, train_fegs, train_label = make_tensors_from_fasta(train_fasta, save_mat_path=train_mat)
    test_pssm,  test_len,  test_fegs,  test_label  = make_tensors_from_fasta(test_fasta,  save_mat_path=test_mat)

    train_loader = DataLoader(
        TensorDataset(train_pssm, train_len, train_fegs, train_label),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(test_pssm, test_len, test_fegs, test_label),
        batch_size=batch_size,
        shuffle=False
    )

    # Model
    model = dvib(k=k, out_channels=out_channels, hidden_size=hidden_size).to(device)
    optimizer = opt.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.95)

    beta = betas[beta_idx]

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0

        if epoch % 10 == 0 and epoch > 0:
            scheduler.step()

        for (sequences, lengths, FEGS, labels) in train_loader:
            # sort by length desc (needed because enforce_sorted=True in pack)
            seq_lengths, perm_idx = lengths.sort(dim=0, descending=True)
            seq_tensor  = sequences[perm_idx].to(device)
            fegs_tensor = FEGS[perm_idx].to(device)
            label       = labels[perm_idx].long().to(device)

            logits, enc_means, enc_stds, latent = model(seq_tensor, seq_lengths, fegs_tensor)
            loss = calc_loss(logits, label, enc_means, enc_stds, beta=beta)

            pred = logits.argmax(dim=1)
            train_correct += pred.eq(label).sum().item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += float(loss.item())

        print(f"Train Epoch:{epoch} Average loss: {train_loss:.6f} "
              f"ACC:{train_correct}/{len(train_loader.dataset)} "
              f"({100.0*train_correct/len(train_loader.dataset):.4f}%)")

        # Evaluate
        model.eval()
        correct = 0
        y_pre, y_true = [], []

        with torch.no_grad():
            for (sequences, lengths, FEGS, labels) in test_loader:
                seq_lengths, perm_idx = lengths.sort(dim=0, descending=True)
                seq_tensor  = sequences[perm_idx].to(device)
                fegs_tensor = FEGS[perm_idx].to(device)
                label       = labels[perm_idx].long().to(device)

                logits, enc_means, enc_stds, latent = model(seq_tensor, seq_lengths, fegs_tensor)

                pred = logits.argmax(dim=1)
                correct += pred.eq(label).sum().item()

                y_true.extend(label.cpu().numpy().tolist())
                y_pre.extend(pred.cpu().numpy().tolist())

        acc = 100.0 * correct / len(test_loader.dataset)
        f1  = metrics.f1_score(y_true, y_pre)
        mcc = metrics.matthews_corrcoef(y_true, y_pre)

        print(f"\nTest: Accuracy:{correct}/{len(test_loader.dataset)} ({acc:.4f}%) "
              f"f1:({100.0*f1:.4f}%) mcc:({100.0*mcc:.4f}%)\n")

    return model


# -------------------------
# 8) Run Example
# -------------------------
if __name__ == "__main__":
    # مسیرها را عوض کن:
    train_fasta = "ToxIBTL-main/Data/training_dataset.fasta"      # مثل فایل خودت
    test_fasta  = "ToxIBTL-main/Data/independent_dataset.fasta"   # مثل فایل خودت

    # اگر می‌خواهی .mat هم بسازد:
    save_mat_dir = "generated_mat"

    model = run_train_test(
        train_fasta=train_fasta,
        test_fasta=test_fasta,
        batch_size=100,
        out_channels=128,
        hidden_size=512,
        k=1024,
        beta_idx=2,
        lr=1e-4,
        num_epochs=50,
        save_mat_dir=save_mat_dir
    )

In [11]:
blosum62 = {
        'A': [4, -1, -2, -2, 0,  -1, -1, 0, -2,  -1, -1, -1, -1, -2, -1, 1,  0,  -3, -2, 0],  # A
        'R': [-1, 5,  0, -2, -3, 1,  0,  -2, 0,  -3, -2, 2,  -1, -3, -2, -1, -1, -3, -2, -3], # R
        'N': [-2, 0,  6,  1,  -3, 0,  0,  0,  1,  -3, -3, 0,  -2, -3, -2, 1,  0,  -4, -2, -3], # N
        'D': [-2, -2, 1,  6,  -3, 0,  2,  -1, -1, -3, -4, -1, -3, -3, -1, 0,  -1, -4, -3, -3], # D
        'C': [0,  -3, -3, -3, 9,  -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1], # C
        'Q': [-1, 1,  0,  0,  -3, 5,  2,  -2, 0,  -3, -2, 1,  0,  -3, -1, 0,  -1, -2, -1, -2], # Q
        'E': [-1, 0,  0,  2,  -4, 2,  5,  -2, 0,  -3, -3, 1,  -2, -3, -1, 0,  -1, -3, -2, -2], # E
        'G': [0,  -2, 0,  -1, -3, -2, -2, 6,  -2, -4, -4, -2, -3, -3, -2, 0,  -2, -2, -3, -3], # G
        'H': [-2, 0,  1,  -1, -3, 0,  0,  -2, 8,  -3, -3, -1, -2, -1, -2, -1, -2, -2, 2,  -3], # H
        'I': [-1, -3, -3, -3, -1, -3, -3, -4, -3, 4,  2,  -3, 1,  0,  -3, -2, -1, -3, -1, 3],  # I
        'L': [-1, -2, -3, -4, -1, -2, -3, -4, -3, 2,  4,  -2, 2,  0,  -3, -2, -1, -2, -1, 1],  # L
        'K': [-1, 2,  0,  -1, -3, 1,  1,  -2, -1, -3, -2, 5,  -1, -3, -1, 0,  -1, -3, -2, -2], # K
        'M': [-1, -1, -2, -3, -1, 0,  -2, -3, -2, 1,  2,  -1, 5,  0,  -2, -1, -1, -1, -1, 1],  # M
        'F': [-2, -3, -3, -3, -2, -3, -3, -3, -1, 0,  0,  -3, 0,  6,  -4, -2, -2, 1,  3,  -1], # F
        'P': [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4, 7,  -1, -1, -4, -3, -2], # P
        'S': [1,  -1, 1,  0,  -1, 0,  0,  0,  -1, -2, -2, 0,  -1, -2, -1, 4,  1,  -3, -2, -2], # S
        'T': [0,  -1, 0,  -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1, 1,  5,  -2, -2, 0],  # T
        'W': [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1, 1,  -4, -3, -2, 11, 2,  -3], # W
        'Y': [-2, -2, -2, -3, -2, -1, -2, -3, 2,  -1, -1, -2, -1, 3,  -3, -2, -2, 2,  7,  -1], # Y
        'V': [0,  -3, -3, -3, -1, -2, -2, -3, -3, 3,  1,  -2, 1,  -1, -2, -2, 0,  -3, -1, 4],  # V
        '-': [0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # -
    }

def normalization(dataset):
    min = dataset.min(axis=0)
    max = dataset.max(axis=0)
    dataset = (dataset - min) / (max - min)
    return dataset

def get_blosum62(seq):
    blosum_list = []
    for i in seq: 
        blosum_list.append(blosum62[i])
    blosum = np.array(blosum_list)
#     blosum = normalization(blosum)
    feature = np.zeros((1002,20))
    idx = blosum.shape[0]
    feature[0:idx,:] = blosum
    return feature


def make_tensor(path):
    data = pd.read_csv(path)
    sequences = data['sequence'].values
    labels = data['label'].values
    evolution = torch.zeros(len(sequences),1002,20)
    lengths = []
    for i in range(len(sequences)):
        lengths.append((len(sequences[i])))
        temp = get_blosum62(sequences[i])
        evolution[i,:,:] = torch.Tensor(temp)

    return evolution,torch.Tensor(lengths),torch.Tensor(labels) 

In [13]:
train_path = 'data/protein_train1002.csv'
test_path = 'data/protein_test1002.csv'

train_data = sio.loadmat('data/benchmark/protein_train.mat')
train_FEGS = torch.Tensor(normalization(train_data['FV']))

test_data = sio.loadmat('data/benchmark/protein_test.mat')
test_FEGS = torch.Tensor(normalization(test_data['FV']))

train_pssm, train_len,train_label = make_tensor(train_path)
test_pssm, test_len,test_label = make_tensor(test_path)

train_data = DataLoader(TensorDataset(train_pssm, train_len,train_FEGS,train_label), batch_size=100, shuffle=True)
test_data = DataLoader(TensorDataset(test_pssm, test_len,test_FEGS, test_label), batch_size=100)

print("data done") 

data done


In [ ]:
# data = pd.read_csv('data/peptide.csv')
# labels = data['label'].values
# sequences = data['sequence'].values
# fegs = sio.loadmat('data/peptide3864.mat')
# X_train, X_test, y_train1, y_test1 = train_test_split(
#     fegs['FV'], labels, test_size=0.15, random_state=66, stratify=labels)

# train_FEGS = torch.Tensor(X_train)
# test_FEGS = torch.Tensor(X_test)

# S_train, S_test, y_train2, y_test2 = train_test_split(
#     sequences, labels, test_size=0.15, random_state=66, stratify=labels)

# train_pssm, train_len,train_label = make_tensor(S_train,y_train2)
# test_pssm, test_len,test_label = make_tensor(S_test,y_test2)

# train_data = DataLoader(TensorDataset(train_pssm, train_len,train_FEGS,train_label), batch_size=100, shuffle=True)
# test_data = DataLoader(TensorDataset(test_pssm, test_len,test_FEGS, test_label), batch_size=100)

In [14]:
class dvib(nn.Module):
    def __init__(self,k,out_channels, hidden_size):
        super(dvib, self).__init__()
        
        self.conv = torch.nn.Conv2d(in_channels=1,
                            out_channels = out_channels,
                            kernel_size = (1,20),
                            stride=(1,1),
                            padding=(0,0),
                            )
        
        self.rnn = torch.nn.GRU(input_size = out_channels,  
                                hidden_size = hidden_size,
                                num_layers = 2,
                                bidirectional = True,
                                batch_first = True,
                                dropout = 0.2
                              )
        
        self.fc1 = nn.Linear(hidden_size*4, hidden_size*4)
#         self.fc2 = nn.Linear(1024,1024)
        self.enc_mean = nn.Linear(hidden_size*4+578,k)
        self.enc_std = nn.Linear(hidden_size*4+578,k)
        self.dec = nn.Linear(k, 2)
        
        self.drop_layer = torch.nn.Dropout(0.5)
        
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.constant_(self.fc1.bias, 0.0)
#         nn.init.xavier_uniform_(self.fc2.weight)
#         nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_mean.weight)
        nn.init.constant_(self.enc_mean.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_std.weight)
        nn.init.constant_(self.enc_std.bias, 0.0)
        nn.init.xavier_uniform_(self.dec.weight)
        nn.init.constant_(self.dec.bias, 0.0)
   
        
        
    def cnn_gru(self,x,lens):
        x = x.unsqueeze(1)
#         print(x.shape)
        x = self.conv(x)
#         print(x.shape)   
        x = torch.nn.ReLU()(x)
#         print(x.shape,type(x))
        x = x.squeeze(3)
#         x = x.view(x.size(0),-1)
        x = x.permute(0,2,1)
#         print(x.shape)
#         print(type(lens))
        gru_input = pack_padded_sequence(x,lens,batch_first=True)
        output, hidden = self.rnn(gru_input)
#         print(hidden.shape)
        output_all = torch.cat([hidden[-1],hidden[-2],hidden[-3],hidden[-4]],dim=1)
#         print("output_all.shape:",output_all.shape)    
        return output_all
    
        
    def forward(self, pssm, lengths, FEGS): 
        cnn_vectors = self.cnn_gru(pssm,lengths)
        feature_vec = torch.cat([cnn_vectors,FEGS],dim = 1)
      
        enc_mean, enc_std = self.enc_mean(feature_vec), f.softplus(self.enc_std(feature_vec)-5)
        eps = torch.randn_like(enc_std)
        latent = enc_mean + enc_std*eps
        
        outputs = f.sigmoid(self.dec(latent))
#         print(outputs.shape)

        return outputs,enc_mean, enc_std,latent

In [15]:
CE = nn.CrossEntropyLoss(reduction='sum')
betas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6,1e-7]

def calc_loss(y_pred, labels, enc_mean, enc_std, beta=1e-3):
    """    
    y_pred : [batch_size,2]
    label : [batch_size,1]    
    enc_mean : [batch_size,z_dim]  
    enc_std: [batch_size,z_dim] 
    """   
    
    ce = CE(y_pred,labels)
    KL = 0.5 * torch.sum(enc_mean.pow(2) + enc_std.pow(2) - 2*enc_std.log() - 1)
    
    return (ce + beta * KL)/y_pred.shape[0]

In [ ]:
out_channels = 128
hidden_size = 512
beta = 2#0，1，3，4，5，6
k = 1024
                
model = dvib(k,out_channels, hidden_size).to(device)
optimizer = opt.Adam(model.parameters(),lr = 0.0001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.95)
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    train_correct = 0
    if epoch % 10 == 0 and epoch > 0:
         scheduler.step()

    for batch_idx, (sequences, lengths, FEGS, labels) in enumerate(train_data):
        seq_lengths, perm_idx = lengths.sort(dim=0,descending=True)
        seq_tensor = sequences[perm_idx].to(device)
        FEGS_tensor = FEGS[perm_idx].to(device)
        label = labels[perm_idx].long().to(device)
#                 seq_lengths = seq_lengths.to(device)

#                 label = torch.LongTensor(label).to(device)

        y_pred,end_means, enc_stds,latent = model(seq_tensor,seq_lengths,FEGS_tensor)
        loss = calc_loss(y_pred, label, end_means, enc_stds, betas[beta])
#             loss = calc_loss(y_pred, label, end_means, enc_stds, betas[2])
#         train_pred = outputs.argmax(dim=1)
        _, train_pred = torch.max(y_pred, 1)
        train_correct += train_pred.eq(label).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() 
    print("Train Epoch:{} Average loss: {:.6f} ACC:{}/{} ({:.4f}%)".format(
                epoch,
                train_loss,
                train_correct, len(train_data.dataset),
                100. * train_correct / len(train_data.dataset)
            ))

    correct = 0
    y_pre = []
    y_test = []
    with torch.no_grad():
        for batch_idx, (sequences, lengths,FEGS, labels) in enumerate(test_data):
            seq_lengths, perm_idx = lengths.sort(dim=0,descending=True)
            seq_tensor = sequences[perm_idx].to(device)
            FEGS_tensor = FEGS[perm_idx].to(device)
            label = labels[perm_idx].long().to(device)
#                     seq_lengths = seq_lengths.to(device)
            y_test.extend(label.cpu().detach().numpy())


            y_pred, end_means, enc_stds,latent = model(seq_tensor,seq_lengths,FEGS_tensor)
            y_pre.extend(y_pred.argmax(dim=1).cpu().detach().numpy())

#             pred = outputs.argmax(dim=1)
#             print(output.shape,label.shape)
            _, pred = torch.max(y_pred, 1) 

            correct += pred.eq(label).sum().item()
#             print(output,label.data)
#             correct += (output == label.data).sum()

        print('\nTest: Accuracy:{}/{} ({:.4f}%) f1:({:.4f}%) mcc:({:.4f}%)\n'.format(
            correct, len(test_data.dataset),
            100. * correct / len(test_data.dataset),
            metrics.f1_score(y_test,y_pre),
            metrics.matthews_corrcoef(y_test,y_pre)
        ))
